# LightGBM con futuro · diario de experimentos (Pablo y Juanma · `futuro/`)

> ⚠️ Usa información de **otros snapshots de la sesión** (pasado y futuro): rompe a propósito la regla 1 del equipo.
> Por eso vive en `futuro/` y guarda en `futuro/oof|preds|submissions`: el stacking normal de `main` no lo ve; solo
> `stacking/stack.py --futuro` lo combina (ver `futuro/README.md`).

**Qué es:** `futuro/m_lgb_future.py`, un LightGBM LambdaRank (parámetros de referencia de `EQUIPO.md`) que puntúa cada tarea con
cuatro bloques de features de `futuro/features_future.py` (ninguna usa la etiqueta):

| Bloque | Qué contiene |
|---|---|
| `base` | columnas originales del snapshot |
| `logic` | reglas dentro del snapshot: plazo por tipo, factibilidad, ventana típica, EDF… (`logic_*`) |
| `past` | cuánto lleva la tarea en la lista, cambios respecto al snapshot anterior, racha de snapshots (`p_*`, `blk_*`, `gap_*`) |
| `future` | segundos hasta la siguiente instrucción al avión y su tipo, si encaja con la tarea, cuándo desaparece la tarea (`f_*`) |

**Cómo usar este notebook** (igual que `NN_pablo.ipynb`)
1. Ejecuta *Preparación* una vez (calcula las features, ~1–2 min).
2. Cada bloque: **hipótesis** → **variantes** (UNA cosa cambia respecto a la base) → **tabla** → conclusión.
3. Lo que gane se combina en `MEJOR`, se confirma con 3 semillas y se guarda para el stacking.

**Cómo leer las cifras:** `cv` = top-3 out-of-fold con `Data/folds.csv`. Pruebas rápidas con 1 semilla: diferencias < ~0.005
son ruido. `min_atco` = peor controlador (el privado es un solo controlador). Referencias: LightGBM limpio de Adri 0.648,
red con futuro 0.887, LightGBM con futuro antiguo (`CLAUDE.md`, folds viejos) 0.83.

**Qué no se repite de `LGB_adri.ipynb`** (solo se lee, no se toca): Adri ya probó, con datos limpios, sus grupos de features
(`snapshot`, `tipo`, `avion`, `tipo_estado`, `contexto`): sueltos aportan poco (+0.001 a +0.004) y **todos juntos empeoran**
(0.648 frente a 0.651). Eso es trabajo suyo y no se repite aquí. Sus **candidatos de parámetros** (`lento`, `regularizado`,
`xendcg`) están escritos pero no los ha lanzado: se prueban aquí con los datos con futuro (bloque 2), para que los dos
LightGBM se comparen con lo mismo.

> ## Punto de partida actual (20/10, noche)
>
> **Mejor modelo: `lgb_future_clear` (CV 0.907).** El suavizado temporal apenas le suma (+0.1 en CV anidada).
>
> | | CV top-3 | peor controlador | dónde |
> |---|---|---|---|
> | `lgb_future` (base: original + logic + past + future) | 0.901 | 0.769 | secciones 0–3; Kaggle público (controlador 020): **0.942** |
> | `lgb_future` + suavizado `prob` (CV anidada) | 0.904 | — | sección 7 |
> | **`lgb_future_clear`** (+ bloque `clear`) | **0.907** | 0.769 | sección 6c (gpu.dsic); comprobado con `cv/metric.py` |
> | `lgb_future_clear` + suavizado `prob` | 0.908 (CV anidada) / 0.910 (parámetros elegidos con todo el OOF) | 0.769 | `futuro/pruebas/oof/suav_lgb_future_clear.csv` |
> | `lgb_future_coord` (+ `coord` + `clear`) | 0.905 | 0.772 | sección 6b |
> | stacking con futuro (`stack.py --futuro`) | 0.903 | — | `futuro/README.md` |
> | XGBoost / CatBoost / red con futuro | 0.896 / 0.874 / 0.887 | 0.731 / 0.689 / 0.746 | 6c / 5 |
>
> **Por qué este:**
> 1. **La información lo es todo:** sin el bloque `future` el LightGBM cae de 0.900 a 0.661 (sección 1). Ningún ajuste de
>    parámetros (2–3), ningún otro modelo (XGBoost, CatBoost, red) ni stacking mejora al LightGBM: con el futuro todos
>    aprenden lo mismo y fallan en los mismos snapshots (5, 6c).
> 2. **`clear` es la única feature nueva que suma de forma robusta:** +0.6 puntos, mejora o iguala los 5 folds, sin empeorar
>    al peor controlador (6c). `coord` no aporta (6b, 6c).
> 3. **El suavizado temporal suma poco y cada vez menos:** +0.3 puntos sobre `lgb_future` (sección 7), pero sobre
>    `lgb_future_clear` solo **+0.1 en CV anidada** (0.9072 → 0.9080): los parámetros buenos cambian de un fold a otro
>    (decay 0.7–1.0, k 2–10), y el 0.910 que sale eligiéndolos con todo el OOF es optimista. Parte de lo que aportaba el
>    suavizado ya lo cuenta el bloque `clear`.
>
> **Para el envío:** `lgb_future_clear` (seguro) o su versión suavizada (`futuro/pruebas/preds/suav_lgb_future_clear.csv`,
> decay=0.85, k=3), con una ganancia esperada de solo ~0.1–0.3 puntos. El gran hueco siguen siendo las coordinaciones
> (entry/exit, ~la mitad de los fallos).

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "futuro") else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "cv")); sys.path.insert(0, str(ROOT / "futuro"))
import m_lgb_future

data = m_lgb_future.build_data()      # una sola vez; todas las pruebas lo reutilizan
REF = {"LGB limpio (Adri)": 0.648, "red con futuro": 0.887}

In [ ]:
# Configuración base = parámetros de referencia de EQUIPO.md con los 4 bloques de features. Cada experimento cambia UNA clave.
BASE = dict(feats="base,logic,past,future", n_estimators=400, lr=0.05, leaves=31, min_child=50, subsample=0.8,
            colsample=0.7, trunc=5, reg_lambda=0.0, objective="lambdarank")


def probar(nombre, motivo, semillas=1, guardar=False, **cambios):
    """Lanza la CV con BASE + cambios; queda apuntado en experiments/lgb_future_runs.csv con su motivo.
    guardar=True genera futuro/oof/, futuro/preds/ y futuro/submissions/ (solo para la versión que va al stacking)."""
    cfg = {**BASE, **cambios}
    args = ["--name", nombre, "--tag", motivo] + (["--quick"] if not guardar else [])
    for k, v in cfg.items():
        args += [f"--{k.replace('_', '-')}", str(v)]
    a = m_lgb_future.parse(args)
    a.seeds = semillas
    return m_lgb_future.run(a, data)


def lanzar(bloque, variantes, semillas=1, sobre=None):
    """variantes: lista de (nombre, motivo, {cambios}). Siempre incluye la base para comparar en igualdad."""
    sobre = sobre or {}
    res = [("base", "referencia", probar(f"{bloque}/base", "referencia", semillas, **sobre))]
    for nombre, motivo, cambios in variantes:
        res.append((nombre, motivo, probar(f"{bloque}/{nombre}", motivo, semillas, **{**sobre, **cambios})))
    return tabla_bloque(res)


def tabla_bloque(res):
    t = pd.DataFrame(res, columns=["variante", "por qué", "cv"])
    ref = t.loc[t.variante == "base", "cv"].iloc[0]
    t["Δ vs base"] = t.cv - ref
    t["veredicto"] = t["Δ vs base"].map(lambda x: "—" if abs(x) < 1e-9 else ("mejora" if x > 0.005 else
                                                   ("empeora" if x < -0.005 else "ruido")))
    return (t.sort_values("cv", ascending=False).style
            .format({"cv": "{:.4f}", "Δ vs base": "{:+.4f}"})
            .background_gradient(subset=["Δ vs base"], cmap="RdYlGn", vmin=-0.02, vmax=0.02).hide(axis="index"))


def historial(filtro=""):
    """Todo lo probado (también desde la terminal), ordenado por CV."""
    r = pd.read_csv(m_lgb_future.LOG)
    r = r[r.name.str.contains(filtro, regex=False)] if filtro else r
    cols = ["when", "name", "tag", "cv", "min_atco", "seeds", "feats", "n_feats", "leaves", "min_child", "lr",
            "n_estimators", "colsample", "secs"]
    return r[[c for c in cols if c in r]].sort_values("cv", ascending=False).reset_index(drop=True)

## 0 · Punto de partida

Base con 3 semillas, generada desde la terminal (`python futuro/m_lgb_future.py`): es la versión `lgb_future` que ya está en
`futuro/oof/` y `futuro/preds/` para Juanma. La celda la mide con la métrica del equipo (`cv/metric.py`).

In [ ]:
from metric import report
report(pd.read_csv(ROOT / "futuro" / "oof" / "lgb_future.csv"), "lgb_future (base, 3 semillas)")

## 1 · ¿Qué bloque de información aporta?

**Hipótesis:** casi todo el salto viene del bloque `future` (la siguiente instrucción delata la tarea); `past` y `logic` aportan poco.
Quitar un bloque cada vez dice cuánto vale cada uno. Si quitar uno **no empeora**, sobra (menos ruido, más robustez).

| Variante | Qué se quita | Qué esperamos |
|---|---|---|
| sin `future` | f_* | caída grande (debería acercarse al LightGBM limpio + pasado) |
| sin `past` | p_*, blk_*, gap_* | caída pequeña |
| sin `logic` | logic_* | ruido: con futuro, las reglas a mano deberían sobrar |

In [ ]:
lanzar("feats", [
    ("sin_future", "¿cuánto aporta el futuro?",       dict(feats="base,logic,past")),
    ("sin_past",   "¿cuánto aporta el pasado?",        dict(feats="base,logic,future")),
    ("sin_logic",  "¿sobran las reglas a mano?",       dict(feats="base,past,future")),
])

**Conclusión bloque 1 · casi toda la información está en el futuro; pasado y reglas no aportan.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | |
|---|---|---|---|---|
| sin `logic` | 0.897 | −0.002 | 0.740 | ruido |
| sin `past` | 0.897 | −0.003 | 0.757 | ruido |
| sin `future` | **0.661** | **−0.239** | 0.596 | empeora muchísimo |

- **El bloque `future` es prácticamente todo el modelo:** sin él cae a 0.661, casi al nivel del LightGBM limpio de Adri (0.648).
- **`past` y `logic` no aportan con el futuro presente** (−0.002/−0.003, ruido): lo que dicen ya lo cuenta mejor la siguiente
  instrucción del controlador. No compensa quitarlos tampoco (no mejora), así que la base se queda con los 4 bloques.
- Sin futuro, `base+logic+past` da 0.661 frente a 0.648 de Adri: ~1 punto, que viene sobre todo del pasado (también rompe la
  regla 1). Las `logic_*` son solo del snapshot: Adri ya las tiene en `features.py` de `main` por si quiere probarlas.
- El peor controlador varía bastante entre variantes (0.740–0.757) con 1 semilla: no sacar conclusiones de ahí.

## 2 · Parámetros (los candidatos de Adri, sobre datos con futuro)

**Hipótesis:** con features tan informativas el modelo se apoya en pocas columnas muy fuertes (`f_next_input`…); lo que
le obligue a repartir (menos columnas por árbol) o a ser más prudente puede generalizar mejor a controladores nuevos.

| Variante | Cambios (iguales que en `LGB_adri.ipynb`) | Por qué |
|---|---|---|
| `lento` | `lr=0.03`, `n_estimators=700` | aprender más despacio con más árboles: suele generalizar mejor |
| `regularizado` | `min_child=100`, `colsample=0.5`, `reg_lambda=5` | hojas más grandes, menos columnas por árbol, pesos más pequeños |
| `xendcg` | `objective="rank_xendcg"` | otra pérdida de ranking, más estable con etiquetas ruidosas |

~1 min por variante (1 semilla).

In [ ]:
lanzar("params", [
    ("lento",        "más lento con más árboles (Adri)",                 dict(lr=0.03, n_estimators=700)),
    ("regularizado", "hojas grandes, menos columnas, L2 (Adri)",         dict(min_child=100, colsample=0.5, reg_lambda=5.0)),
    ("xendcg",       "otra pérdida de ranking (Adri)",                   dict(objective="rank_xendcg")),
])

**Conclusión bloque 2 · la CV no se mueve, pero `regularizado` y `lento` mejoran al peor controlador.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | Δ peor | |
|---|---|---|---|---|---|
| lento (`lr=0.03`, 700 árboles) | 0.900 | +0.001 | 0.763 | +0.018 | CV igual, más robusto |
| regularizado (`min_child=100`, `colsample=0.5`, `reg_lambda=5`) | 0.900 | 0.000 | **0.775** | **+0.030** | CV igual, más robusto |
| xendcg | 0.886 | **−0.013** | 0.740 | −0.006 | empeora |

- **En CV media ningún candidato gana:** con el futuro, el modelo base ya está bien ajustado.
- **El peor controlador sí mejora** con `regularizado` (+3 puntos) y `lento` (+1,8). Importa porque el leaderboard privado
  es **un solo controlador** (023) que no está en train: un modelo que no se hunda con ninguno es más seguro. Encaja con la
  hipótesis: obligarle a repartir entre columnas (`colsample=0.5`) evita que dependa solo de `f_next_input`.
- Ojo: con 1 semilla el peor controlador baila (en el bloque 1 iba de 0.740 a 0.757), así que hay que confirmarlo con 3 semillas.
- **`xendcg` se descarta.**

## 3 · Combinar lo que ha funcionado

Pon en `MEJOR` los cambios con veredicto **mejora** de los bloques 1 (p. ej. `feats=...` si sobra un bloque) y 2. Se comprueba con **3 semillas** frente a la base de 3 semillas
(sección 0), porque varios cambios que ayudan por separado no siempre suman.

In [ ]:
MEJOR = dict(
    # bloque 2: misma CV, mejor peor controlador -> se confirma con 3 semillas en la celda siguiente
    min_child=100, colsample=0.5, reg_lambda=5.0,     # regularizado
    lr=0.03, n_estimators=700,                        # lento
)

In [ ]:
# Comprobación con 3 semillas. Se puede saltar: el bloque 4 solo necesita la celda de MEJOR.
cv_comb = probar("comb/" + ",".join(f"{k}={v}" for k, v in MEJOR.items()), "combinación de ganadores", semillas=3, **MEJOR)
print(f"combinación: {cv_comb:.4f}   referencias: {REF}")

**Conclusión bloque 3 · la combinación no mejora: la mejora del peor controlador era ruido de 1 semilla.**

| (3 semillas) | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| **`lgb_future`** (base) | **0.901** | **0.769** | 0.938 | 0.890 | 0.879 | 0.896 | 0.904 |
| `MEJOR` (regularizado + lento) | 0.901 | 0.766 | 0.938 | 0.894 | 0.884 | 0.885 | 0.904 |

- Misma CV y mismo peor controlador: con 3 semillas desaparece la ventaja que se veía con 1 (0.775 vs 0.746).
- Además tarda el doble (700 árboles). **Se queda `lgb_future` como versión final** y el bloque 4 no se lanza.
- Lectura general del barrido: con el futuro, los parámetros de referencia de `EQUIPO.md` ya están bien; el modelo depende de
  la información (bloque 1), no del ajuste.

## 4 · Versión para el stacking (`futuro/`)

Si la combinación mejora a la base, genera la versión definitiva con 3 semillas en el formato común. Nombre nuevo
(`lgb_future_v2`…) para no pisar `lgb_future`. Juanma la combina con `stack.py --futuro`.

In [ ]:
NOMBRE = "lgb_future_v2"
probar(NOMBRE, "versión stacking futuro: " + str(MEJOR), semillas=3, guardar=True, **MEJOR)

**Resultado:** no se lanza; `lgb_future` (CV 0.901, ya en `futuro/oof/` y `futuro/preds/`) es la versión para el stacking de esta rama.

## 5 · Análisis de errores y comparación con la red con futuro

**Pregunta:** ¿fallan el LightGBM y la red en los mismos snapshots? Si no, la combinación de Juanma ganará. Se mira el acierto
por tipo de tarea de cada modelo y en cuántos snapshots acierta uno solo.

In [ ]:
from metric import hits, labels

TIPOS = ["entry_conflict", "exit_conflict", "entry_coord", "exit_coord", "assume", "transfer", "conflict",
         "non_conformance", "qos", "entry_conditions", "exit_conditions", "aircraft_request", "zone", "return_to_route"]


def comparar(modelos=("lgb_future", "nn_future")):
    lab = labels().set_index("sample_id")
    tipos = pd.concat([pd.read_csv(ROOT / "Data" / f"{f}.csv", usecols=["sample_id", "target_index", "task_index"])
                       for f in ("train", "validation")]).query("target_index == 1").set_index("sample_id").task_index
    h = pd.DataFrame({m: hits(pd.read_csv(ROOT / "futuro" / "oof" / f"{m}.csv")) for m in modelos})
    a, b = modelos
    print(f"top-3:  {a} {h[a].mean():.3f}   {b} {h[b].mean():.3f}")
    print(f"aciertan los dos {(h[a] & h[b]).mean():.3f} | solo {a} {(h[a] & ~h[b].astype(bool)).mean():.3f} | "
          f"solo {b} {(~h[a].astype(bool) & h[b]).mean():.3f} | ninguno {((h[a] == 0) & (h[b] == 0)).mean():.3f}")
    h["tipo"] = tipos.reindex(h.index).map(dict(enumerate(TIPOS)))
    t = h.groupby("tipo").agg(muestras=(a, "size"), **{a: (a, "mean"), b: (b, "mean")})
    t["Δ (lgb − red)"] = t[a] - t[b]
    return t.sort_values("muestras", ascending=False).style.format(
        {a: "{:.2f}", b: "{:.2f}", "Δ (lgb − red)": "{:+.2f}"}).background_gradient(subset=["Δ (lgb − red)"], cmap="RdYlGn")


comparar(("lgb_future", "nn_future"))      # cambiar a lgb_future_v2 si se genera

**Conclusión del análisis · los dos modelos fallan casi en los mismos sitios; mezclarlos a partes iguales no suma.**

| | % de snapshots |
|---|---|
| aciertan los dos | 86,6 % |
| solo `lgb_future` | 3,5 % |
| solo `nn_future` | 2,2 % |
| fallan los dos | 7,7 % |

| Tipo correcto | muestras | `lgb_future` | `nn_future` | Δ |
|---|---|---|---|---|
| transfer | 3.527 | 0.98 | 0.98 | +0.00 |
| assume | 3.366 | 0.98 | 0.98 | +0.01 |
| zone_conflict | 2.121 | 0.92 | 0.90 | +0.02 |
| entry_coordination | 1.068 | 0.71 | 0.67 | +0.03 |
| return_to_route | 937 | 0.87 | 0.82 | +0.04 |
| exit_coordination | 745 | 0.57 | **0.59** | **−0.02** |
| exit_conditions | 448 | 0.80 | 0.76 | +0.04 |

- El LightGBM es igual o mejor en casi todos los tipos; la red solo gana en **exit_coordination**.
- Media de puntuaciones normalizadas (z-score dentro del snapshot), según el peso de la red: 0 → **0.901**, 0.3 → 0.900,
  0.5 → 0.897, 1 → 0.887. **Cuanto más peso a la red, peor:** con la información del futuro, los dos aprenden lo mismo.
- Para Juanma: con el futuro, `lgb_future` solo ya es lo mejor. Su meta-modelo (que puede fiarse de la red solo en
  exit_coordination) podría arañar algo, pero no hay que esperar mucho. Contraste con los modelos limpios, donde la red sí
  sumaba (LightGBM 0.648 → mezcla 0.655): sin futuro hay menos información y cada modelo captura cosas distintas.
- (Cifras calculadas con la misma función de la celda anterior; la salida de la celda no quedó guardada en el notebook.)

## 5b · Suavizado temporal de las predicciones

**Hipótesis:** la descripción de los datos dice que los snapshots consecutivos suelen compartir respuesta (el 72 % repite la del
anterior dentro de una racha contigua; rachas de 3 snapshots de mediana). Promediar la probabilidad de cada tarea con la de
la misma tarea en los snapshots vecinos (pesos que decaen con la distancia) debería corregir fallos aislados.

In [ ]:
import numpy as np, features_future as ff

_blk = ff.add_blocks(ff.add_folds(ff.load()))
_blk = _blk[_blk.split != "test"][["sample_id", "row_index", "task_name", "block", "blk_pos"]]


def suavizar(modelo, k=5, decay=0.7):
    """Media ponderada de la probabilidad de la misma tarea en los ±k snapshots contiguos."""
    x = _blk.merge(pd.read_csv(ROOT / "futuro" / "oof" / f"{modelo}.csv"), on=["sample_id", "row_index"])
    x["p"] = np.exp(x.score - x.groupby("sample_id").score.transform("max"))
    x["p"] /= x.groupby("sample_id").p.transform("sum")
    x = x.sort_values(["block", "task_name", "blk_pos"]); g = x.groupby(["block", "task_name"]).p
    num, den = x.p.copy(), pd.Series(1.0, index=x.index)
    for i in range(1, k + 1):
        for sh in (g.shift(i), g.shift(-i)):
            num += decay ** i * sh.fillna(0); den += decay ** i * sh.notna()
    return hits(x[["sample_id", "row_index"]].assign(score=x.p)).mean(), hits(x[["sample_id", "row_index"]].assign(score=num / den)).mean()


pd.DataFrame({m: suavizar(m) for m in ("lgb_future", "nn_future")}, index=["sin suavizar", "suavizado k=5"]).round(4)

**Conclusión 5b:** mejora real pero pequeña (calculado con esta función, k = 2 / 5 / 10):

| | sin suavizar | k=2 | k=5 | k=10 |
|---|---|---|---|---|
| `lgb_future` | 0.901 | 0.906 | 0.906 | **0.907** |
| `nn_future` | 0.887 | 0.895 | **0.898** | 0.898 |

- +0.6 puntos en el LightGBM, +1 en la red: las features de futuro ya contienen buena parte de esa información.
- Para usarlo en test haría falta aplicarlo también a `preds/` (mismas rachas en las sesiones de test). Candidato para la
  versión final de esta rama.

## 6 · Bloques nuevos: `coord` y `clear`

**Por qué:** en la sección 5 lo que queda son sobre todo las **coordinaciones** (entry + exit ≈ 45 % de los fallos) y
`f_next_input` no las ve: coordinar con un sector vecino no es una instrucción al avión. Los dos bloques nuevos miden,
para cada tarea, **cuántos segundos faltan hasta que cambia en la sesión el estado que esa tarea resuelve**:

| Bloque | Archivo | Columnas | Qué cambio futuro mide | Tipos a los que apunta |
|---|---|---|---|---|
| `coord` | `futuro/feat_coord.py` | `sc_*` | coordinación de entrada (`x_isc_*`), de salida (`k_out_*`), assume (`is_assumed` 0→1), transfer (1→0) | entry/exit_coordination, assume, transfer, gemelo assume/entry |
| `clear` | `futuro/feat_clear.py` | `sx_*` | nivel autorizado, rumbo, nivel de salida, fin de zona/conflicto, vuelta a ruta | exit_conditions, zone_conflict, return_to_route, conflictos |

**Hipótesis:** `coord` debería subir sobre todo entry/exit_coordination; `clear` los tipos de instrucciones. Se añaden de
uno en uno sobre la base (los 4 bloques de siempre), y después juntos. 1 semilla, ~1 min por variante.

In [ ]:
lanzar("nuevos", [
    ("+coord",       "cambios futuros de coordinación/assume/transfer",  dict(feats="base,logic,past,future,coord")),
    ("+clear",       "cambios futuros de instrucciones",                 dict(feats="base,logic,past,future,clear")),
    ("+coord+clear", "los dos bloques nuevos",                           dict(feats="base,logic,past,future,coord,clear")),
])

**Conclusión bloque 6 · `clear` mejora; `coord`, sorprendentemente, casi no.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | |
|---|---|---|---|---|
| +clear | **0.906** | **+0.006** | 0.757 | mejora (justo por encima del umbral) |
| +coord+clear | 0.905 | +0.006 | 0.775 | mejora |
| +coord | 0.901 | +0.001 | 0.769 | ruido |

- **`clear` es la que aporta:** sube 4 de los 5 folds (fold 0 +0.012, fold 2 +0.023). Saber cuándo cambia el nivel, el rumbo o
  la ruta del avión delata las tareas de instrucciones (exit_conditions, zone, return_to_route).
- **`coord` sola no mueve la CV**, aunque iba a por las coordinaciones. Lo más probable es que esa información ya estuviera en
  `f_task_end` (la tarea de coordinación desaparece de la lista justo cuando se coordina).
- `+coord+clear` empata con `+clear` en CV y tiene mejor peor controlador (0.775), pero con 1 semilla ese número baila
  (ya pasó en el bloque 2). La comparación por tipo de 6b dirá si `coord` aporta algo en las coordinaciones.
- **6b se lanza con `coord+clear`**: si por tipo `coord` no mejora las coordinaciones, la versión final se queda solo con `clear`.

### 6b · Versión con los bloques que ganen y dónde mejora

Poner en `FEATS_NUEVAS` la combinación ganadora del bloque 6. Se genera con 3 semillas como **`lgb_future_coord`**
(no pisa `lgb_future`) y se compara por tipo de tarea con `lgb_future`: así se ve si mejora justo donde se esperaba
(coordinaciones / instrucciones). ~4 min.

In [ ]:
FEATS_NUEVAS = "base,logic,past,future,coord,clear"     # bloque 6: +clear y +coord+clear mejoran (+0.006)
probar("lgb_future_coord", "versión stacking futuro: " + FEATS_NUEVAS, semillas=3, guardar=True, feats=FEATS_NUEVAS)
comparar(("lgb_future_coord", "lgb_future"))


=== lgb_future_coord versión stacking futuro: base,logic,past,future,coord,clear  seeds=3 feats=base,logic,past,future,coord,clear n_estimators=400 lr=0.05 leaves=31 min_child=50 subsample=0.8 colsample=0.7 trunc=5 reg_lambda=0.0 objective=lambdarank folds= quick=False
  fold 0 (37s)
  fold 1 (73s)
  fold 2 (109s)
  fold 3 (145s)
  fold 4 (181s)
lgb_future_coord: CV top3 = 0.9045   (min atco 0.772)   f0:0.950  f1:0.892  f2:0.887  f3:0.894  f4:0.903   147 features
Guardado: lgb_future_coord (oof) (preds + sub)
top-3:  lgb_future_coord 0.905   lgb_future 0.901
aciertan los dos 0.888 | solo lgb_future_coord 0.017 | solo lgb_future 0.014 | ninguno 0.082


,muestras,lgb_future_coord,lgb_future,Δ (lgb − red)
tipo,,,,
transfer,3527,0.98,0.98,+0.00
assume,3366,0.98,0.98,+0.00
zone,2121,0.92,0.92,-0.00
entry_coord,1068,0.73,0.71,+0.03
return_to_route,937,0.87,0.87,-0.00
exit_coord,745,0.59,0.57,+0.02
exit_conditions,448,0.82,0.80,+0.02
conflict,390,0.86,0.85,+0.01
qos,257,0.84,0.83,+0.01


**Conclusión 6b · ganancia pequeña pero real, y justo donde se esperaba.**

| 3 semillas | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| `lgb_future` | 0.901 | 0.769 | 0.938 | 0.890 | 0.879 | 0.896 | 0.904 |
| **`lgb_future_coord`** | **0.905** | 0.772 | 0.950 | 0.892 | 0.887 | 0.894 | 0.903 |

| Tipo | `lgb_future` | `lgb_future_coord` | Δ |
|---|---|---|---|
| entry_coordination | 0.706 | 0.732 | **+0.026** |
| exit_coordination | 0.569 | 0.592 | **+0.023** |
| exit_conditions | 0.799 | 0.821 | **+0.022** |
| zone_conflict | 0.920 | 0.916 | −0.004 |
| return_to_route | 0.868 | 0.866 | −0.002 |
| non_conformance (86 muestras) | 0.895 | 0.791 | −0.105 |

- **+0.35 puntos** con 3 semillas (la mitad de lo que sugería 1 semilla); mejoran 3 de 5 folds.
- Las coordinaciones suben más de 2 puntos frente a `lgb_future`, **pero eso no demuestra que sea por `coord`**: la
  comparación es contra la base, no contra `+clear` solo. La sección 6c (con `+clear` solo a 3 semillas) indica que
  `coord` no aporta e incluso resta un poco.
- `clear` mejora exit_conditions, pero no zone_conflict ni return_to_route, a las que también apuntaba.
- non_conformance cae 10 puntos, pero son 86 muestras (~9 aciertos): probablemente ruido; vigilarlo.
- ~~`lgb_future_coord` pasa a ser el mejor LightGBM con futuro~~ → lo supera `lgb_future_clear` (0.907, sección 6c).
  `lgb_future_coord` es el mismo modelo que en el estudio de gpu.dsic se llamó `lgb_future_v2` (mismas cifras).
- Pendiente (norma de probar en las dos redes): añadir `sc_*` y `sx_*` a la red con futuro.

## 6c · Estudio en gpu.dsic (20/10, 21:35–22:20): otros árboles y confirmación de `clear`

Lanzado desde la terminal en gpu.dsic con `futuro/m_gbm_future.py` y `futuro/m_lgb_future.py`, mismos folds y features.
Referencia: `lgb_future` = CV 0.901 (3 semillas), 0.8996 (1 semilla). **En Kaggle (público, controlador 020): 0.942.**
Sus salidas y registros se generaron en gpu.dsic y no están en el repo: aquí solo quedan las cifras.

**1 · ¿Otro modelo de árboles mejora al LightGBM? → No** (mismas 125 features, 3 semillas)

| Modelo | CV top-3 | Peor controlador | Folds |
|---|---|---|---|
| **lgb_future** (referencia) | **0.901** | **0.769** | 0.938 / 0.890 / 0.879 / 0.896 / 0.904 |
| XGBoost `rank:ndcg` (600 árboles, depth 6, GPU) | 0.896 | 0.731 | 0.929 / 0.881 / 0.891 / 0.887 / 0.896 |
| CatBoost `YetiRank` (1500 iteraciones, depth 6, GPU) | 0.874 | 0.689 | 0.905 / 0.868 / 0.848 / 0.875 / 0.876 |

| Stacking (`stack.py --futuro`) | Mejor stack | Peor controlador |
|---|---|---|
| lgb + xgb | 0.9010 (los pesos dan 0 a xgb) | 0.769 |
| lgb + xgb + cat | 0.9012 (`stack_avg` por rangos) | 0.754 |

Correlación de aciertos: lgb–xgb 0.86, lgb–cat 0.73. Ni solos ni combinados mejoran al LightGBM, y empeoran al peor
controlador. Igual que con la red con futuro (sección 5): **con la información del futuro todos los modelos aprenden lo
mismo; el límite está en la información, no en el modelo.** XGBoost y CatBoost se descartan.

**2 · `clear` confirmado con 3 semillas**

| Modelo | CV | Δ vs 0.901 | Peor controlador | Folds |
|---|---|---|---|---|
| **`lgb_future_clear`** (+clear) | **0.9072** | **+0.006** | 0.769 | 0.949 / 0.893 / 0.894 / 0.897 / 0.905 |
| `lgb_future_coord` (+coord +clear, sección 6b) | 0.9045 | +0.004 | 0.772 | 0.950 / 0.892 / 0.887 / 0.894 / 0.903 |

- **`clear` funciona:** +0.6 puntos, mejora o iguala en los 5 folds y no empeora al peor controlador.
- **`coord` no aporta, e incluso resta un poco** combinado con `clear`. Las coordinaciones (la mitad de los fallos que
  quedan) siguen sin verse. Hipótesis: el cambio de estado de coordinación ocurre lejos en el tiempo de cuando el
  controlador trabaja en ella, o cae en huecos sin snapshots etiquetados.
- **Modelo elegido: `lgb_future_clear` (CV 0.907)**, base del segundo nivel temporal (suavizado, sección 7)
  y del envío de las ~00:00.

**3 · Notas técnicas de gpu.dsic:** 2× Quadro RTX 5000 (16 GB), 32 núcleos. LightGBM va en CPU (8 hilos): caben 4 pruebas
a la vez. XGBoost 3.x de pip no trae GPU allí → `xgboost==2.0.3` con `task_index` en one-hot (su camino categórico falla
con numpy 2). CatBoost YetiRank en GPU calcula la métrica PFound en CPU: se calcula solo al final (`metric_period`).

## 7 · Suavizado temporal con test y CV anidada (Juanma)

Completa la 5b (mismo método `prob`, ahora con CV anidada y aplicado a test). Hay un snapshot cada ~3 s y el controlador suele seguir con la misma tarea varios seguidos, así que se mezcla
el score de cada tarea con el de la **misma tarea** (`session`, `task_name`) en los ±k snapshots de la misma racha (`block`):

- **prob** (el de la 5b, de Pablo): `(p(t) + Σ_j decay^j · p(t±j)) / (1 + Σ_j decay^j · [la tarea está en ese vecino])`, con `p` = softmax del score dentro del snapshot.
- **log** (alternativa): `lsm(t) + a · Σ_j decay^(j-1) · [lsm(t-j) + lsm(t+j)]`, con `lsm` = log-softmax.

Se comparan los dos con CV anidada y se usa el que gane para cada modelo.

Diferencias con la 5b:
- **CV anidada**: `a`, `decay` y `k` se eligen con 4 folds y se miden en el 5º. Así el número no está inflado por elegir entre muchas combinaciones.
- **Se aplica también a test** (`preds/`), con los vecinos de las sesiones de test. No usa etiquetas, así que no hay fuga. Los vecinos son de la misma sesión, por tanto del mismo controlador y fold, y su OOF tampoco vio la etiqueta.
- Las salidas van a `futuro/pruebas/oof|preds/suav_<modelo>.csv` y salen en `futuro/pruebas/panel_pruebas.ipynb`.

Código en `futuro/pruebas/suavizado.py`.

In [ ]:
import itertools
sys.path.insert(0, str(ROOT / "futuro" / "pruebas"))
import suavizado as S
from base_fut import datos, guardar, registrar

MODELOS = ["lgb_future", "nn_future"]          # cualquiera con oof (y preds) en futuro/ o futuro/pruebas/
GRIDS = {"prob": list(itertools.product((0.5, 0.7, 0.85, 1.0), (1, 2, 3, 5, 10))),                       # (decay, k)
         "log": list(itertools.product((0.1, 0.2, 0.3, 0.5, 0.75, 1.0), (1.0, 0.7, 0.5), (1, 2, 3, 5, 10)))}  # (a, decay, k)

_lab, _te, _ = datos()
_cols = ["sample_id", "row_index", "session", "block", "t", "task_name"]
LS = _lab[_cols + ["target_index", "fold"]].assign(y=_lab.target_index)
TS = _te[_cols].copy()
NB_L, NB_T = S.vecinos(LS, 10), S.vecinos(TS, 10)   # posición de la misma tarea en los ±10 snapshots vecinos

In [ ]:
def aplicar(metodo, score_lsm, nb, g):
    return S.suavizar_prob(np.exp(score_lsm), nb, *g) if metodo == "prob" else S.suavizar(score_lsm, nb, *g)


def suavizado_anidado(modelo):
    L = LS.merge(S.leer(modelo, "oof"), on=["sample_id", "row_index"], how="left")
    base = S.lsm(L, L.score.values)
    n, mejor = L.sample_id.nunique(), None
    for metodo, grid in GRIDS.items():
        sc = {g: aplicar(metodo, base, NB_L, g) for g in grid}
        filas, tot0, tot1 = [], 0, 0
        for f in range(5):
            tr, va = (L.fold != f).values, (L.fold == f).values
            g = max(grid, key=lambda g: S.hit3(L, sc[g], tr))
            h0, h1, m = S.hit3(L, base, va), S.hit3(L, sc[g], va), L[va].sample_id.nunique()
            tot0, tot1 = tot0 + h0 * m, tot1 + h1 * m
            filas.append(dict(fold=f, params=g, sin=h0, con=h1, mejora=h1 - h0))
        g_final = max(grid, key=lambda g: S.hit3(L, sc[g]))
        print(f"{modelo} · {metodo}: CV anidada {tot0 / n:.4f} -> {tot1 / n:.4f} ({(tot1 - tot0) / n:+.4f}); "
              f"parámetros finales (todo el OOF): {g_final}")
        display(pd.DataFrame(filas).round(4))
        if mejor is None or tot1 > mejor[0]:
            mejor = (tot1, metodo, g_final)
    print(f"-> {modelo}: se usa {mejor[1]} con {mejor[2]}\n")
    return L, base, mejor[1], mejor[2]


SUAV = {m: suavizado_anidado(m) for m in MODELOS}

In [ ]:
# aplica el método y los parámetros finales a OOF y test, guarda suav_<modelo> y lo registra para el panel
for modelo, (L, base, metodo, g) in SUAV.items():
    p = S.leer(modelo, "preds")
    T = TS.merge(p, on=["sample_id", "row_index"], how="left") if p is not None else None
    so = aplicar(metodo, base, NB_L, g)
    sp = aplicar(metodo, S.lsm(T, T.score.values), NB_T, g) if T is not None else None
    guardar(f"suav_{modelo}", L, so, T, sp)
    registrar(f"suav_{modelo}", hilo="suavizado", n_features=None,
              descripcion=f"{modelo} suavizado {metodo} {g} (notebook LGB_futuro)",
              extra={"metodo": metodo, "params": list(g)})
    print(f"suav_{modelo}: guardado", "con preds de test" if sp is not None else "SIN preds (no hay preds del modelo)")

**Conclusión 7** (CV anidada, primera medida):

| | sin suavizar | log | **prob (5b)** |
|---|---|---|---|
| `lgb_future` | 0.9010 | 0.9033 (+0.0023) | **0.9042 (+0.0032)** |
| `nn_future` | 0.8874 | 0.8776 (−0.0097) | **0.8973 (+0.0099)** |

- El método de la 5b (media de probabilidades normalizada) es el bueno. En la red el log empeora, porque sus scores tienen otra escala.
- Los 0.906–0.907 de la 5b salían de elegir k mirando todo el OOF. En CV anidada la mejora real es +0.3 puntos en el LightGBM y +1 en la red.
- Con preds de test ya se puede usar: siguiente paso, suavizar el stack de futuro.

## 8 · Siguientes ideas

- Las coordinaciones siguen siendo el gran hueco: buscar otra señal (p. ej. huecos sin snapshots etiquetados alrededor
  del cambio de coordinación).
- Probar `clear` en la red con futuro (norma de las dos redes).
- Si Juanma ve que la combinación `lgb_future` + `nn_future` gana, afinar el peso de cada uno en su `stack.py`.

## Historial completo

In [ ]:
historial()